# Step2_3_B_C_D cyclic integration

큰 흐름은 다음 4단계 사이클입니다.

1. Binary 질의를 활용한 multi-labeling
2. Granularity estimation and sorting
3. Intersection-based integration
4. New phrase generation and phrase list update

첫 사이클은 기존 Step1 binary/intersection 결과를 사용합니다. 두 번째 사이클부터는 이전 binary 결과를 재사용하고, 새로 추가된 phrase에 대해서만 binary 질의를 수행한 뒤 long-tail 처리 없이 바로 intersection matrix를 만듭니다.

In [6]:
import pandas as pd
df_test = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Train/MINT/RE_TACRED_train_phrase.csv')
df_test.head()

,split,sample_id,text,token,subject,subj_start,subj_end,subj_type,object,obj_start,obj_end,obj_type,relation,gpt4_phrase,gpt4_phrase_preprocessed
0,train,61b3a65fb9acc5c72e88,"* `` Updates , Dcr -LRB- DCR -RRB- '' < Dcr.Up...","['*', '``', 'Updates', ',', 'Dcr', '-LRB-', 'D...",DCR,6,6,ORGANIZATION,Dcr,4,4,ORGANIZATION,org:alternate_names,alias,alias
1,train,61b3a65fb916e324f16c,-LRB- While I 'm sure that Fidelity also reser...,"['-LRB-', 'While', 'I', ""'m"", 'sure', 'that', ...",Fidelity,14,14,ORGANIZATION,Fidelity,6,6,ORGANIZATION,org:alternate_names,self-reference,self-reference
2,train,61b3a65fb96a377b22eb,"In my case , it goes something along the lines...","['In', 'my', 'case', ',', 'it', 'goes', 'somet...",Pentax,47,47,ORGANIZATION,Pentax,19,19,ORGANIZATION,org:alternate_names,brand-product,brand-product
3,dev,e7798a9e3660ae176967,The documents on ghostwriting were uncovered b...,"['The', 'documents', 'on', 'ghostwriting', 'we...",Public Library of Science,28,31,ORGANIZATION,PLoS,20,20,ORGANIZATION,org:alternate_names,organization_publisher_of_journal,organization_publisher_of_journal
4,train,61b3a65fb997f51a5aef,Zhang Chunxian has been appointed as the Party...,"['Zhang', 'Chunxian', 'has', 'been', 'appointe...",Communist Party of China,21,24,ORGANIZATION,CPC,26,26,ORGANIZATION,org:alternate_names,alias,alias


In [ ]:

from pathlib import Path

BASE_DIR = Path("/workspace/KG-known0/ARR2026/Data_paper/RE-TACRED/Train/MINT")

DATA_PATH = BASE_DIR / "RE_TACRED_train_phrase.csv"
INITIAL_BINARY_PATH = BASE_DIR / "RE_TACRED_full_binary_result.csv"
INITIAL_PREDICATE_PATH = BASE_DIR / "RE_TACRED_full_binary_question_result_predicate.csv"
INITIAL_INTERSECTION_PATH = BASE_DIR / "RE_TACRED_full_intersection_matrix.csv"

OUTPUT_DIR = Path("/workspace/KG-known0/ARR2026/MINT-ORE/RE_TACRED/output")
log_path = OUTPUT_DIR / "RE_TACRED_full_cycle_log.txt"


MAX_CYCLES = None  
STOP_WHEN_NO_NEW_PHRASES = True


FIXED_TOP_RATIO = 0.1




GRANULARITY_ANCHOR = 0.5


INTEGRATION_THRESHOLD = 0.7 


OPENAI_MODEL = "gpt-4.1-mini"
API_TEMPERATURE = 0.0
BINARY_MAX_WORKERS = 10
BINARY_SAVE_EVERY = 410


RUN_ID_PREFIX = "Cycle"
print("BASE_DIR:", BASE_DIR)

BASE_DIR: /workspace/KG-known0/ARR2026/Data_paper/TACRED/Train/MINT


In [2]:

import ast
import json
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None

if client is None:
    print("[WARN] OPENAI_API_KEY가 없어 API 호출 셀 실행 시 오류가 납니다.")

In [3]:

def ensure_list(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
            return [v]
        except Exception:
            return [s]
    return [x]


def unique_keep_order(values):
    out = []
    seen = set()
    for v in values:
        if v is None or pd.isna(v):
            continue
        v = str(v).strip()
        if not v or v in seen:
            continue
        seen.add(v)
        out.append(v)
    return out


def save_list(values, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    values = unique_keep_order(values)
    path.write_text(str(values), encoding="utf-8")
    print(f"Saved list: {path} ({len(values)})")
    return values


def load_list(path, default=None):
    path = Path(path)
    if not path.exists():
        return [] if default is None else default
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return [] if default is None else default
    return ast.literal_eval(text)


def relation_columns_from_intersection(labels_count):
    relations = set(labels_count["predicted_relation"].astype(str))
    return [c for c in labels_count.columns if c in relations]

In [4]:

def build_binary_prompt(sentence: str, subject: str, obj: str, candidate_relation: str) -> str:
    return (
        "You are an expert annotator for relation extraction tasks.\n"
        "Your task consists of three steps.\n"
        "1. Identify the relation between the subject and the object within the given sentence.\n"
        "2. Determine whether the relation between the subject and the object belongs to the given candidate relation class.\n"
        "3. Output 1 if it belongs, and 0 if it does not.\n"
        "Do not output anything other than the classification result (0 or 1).\n"
        "Now, review the sample below and perform the classification.\n\n"
        f"Sentence: {sentence}\n"
        f"Subject: {subject}\n"
        f"Object: {obj}\n"
        f"Candidate relation: {candidate_relation}\n\n"
    )


def parse_binary_answer(text):
    if text is None:
        return 0
    for ch in str(text).strip():
        if ch in ("0", "1"):
            return int(ch)
    return 0

import time
import pandas as pd

def call_binary_api(idx: int, row: pd.Series, candidate_relation: str):
    if client is None:
        raise RuntimeError("OPENAI_API_KEY가 설정되지 않았습니다.")

    prompt = build_binary_prompt(
        row["text"],
        row["subject"],
        row["object"],
        candidate_relation
    )

    max_retries = 3

    
    for attempt in range(max_retries + 1):
        try:
            resp = client.responses.create(
                model=OPENAI_MODEL,
                input=prompt,
                temperature=API_TEMPERATURE
            )
            return idx, parse_binary_answer(resp.output_text)

        except Exception as e:
            print(
                f"[ERROR] API 호출 실패 "
                f"(index={idx}, relation={candidate_relation}, attempt={attempt + 1}/{max_retries + 1}): {e}"
            )

            
            if attempt < max_retries:
                continue

    
    print(f"[WAIT] index={idx}, relation={candidate_relation}: 10초 대기 후 마지막 재시도")
    time.sleep(10)

    try:
        resp = client.responses.create(
            model=OPENAI_MODEL,
            input=prompt,
            temperature=API_TEMPERATURE
        )
        return idx, parse_binary_answer(resp.output_text)

    except Exception as e:
        print(
            f"[FINAL ERROR] API 최종 호출 실패 "
            f"(index={idx}, relation={candidate_relation}): {e}"
        )
        return idx, "ERROR"


def run_binary_query_for_new_relations(df_result, candidate_relations, binary_before_df, save_path):
    sampled_df = df_result.copy()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    candidate_relations = unique_keep_order(candidate_relations)

    if "multi_labels" not in sampled_df.columns:
        sampled_df["multi_labels"] = [[] for _ in range(len(sampled_df))]
    else:
        sampled_df["multi_labels"] = sampled_df["multi_labels"].apply(ensure_list)

    binary_before_df = binary_before_df.copy()
    if "multi_labels" in binary_before_df.columns:
        binary_before_df["multi_labels"] = binary_before_df["multi_labels"].apply(ensure_list)

    if "id" not in sampled_df.columns or "id" not in binary_before_df.columns:
        raise ValueError("sampled_df와 binary_before_df 모두에 'id' 열이 있어야 합니다.")

    before_by_id = binary_before_df.drop_duplicates(subset=["id"]).set_index("id")
    before_cols = set(before_by_id.columns)

    reused_relations = [r for r in candidate_relations if r in before_cols]
    new_relations = [r for r in candidate_relations if r not in before_cols]

    print(f"[INFO] 이전 결과 재사용 relation 수: {len(reused_relations)}")
    print(f"[INFO] 새로 API 호출할 relation 수: {len(new_relations)}")

    for relation_name in reused_relations:
        sampled_df[relation_name] = sampled_df["id"].map(before_by_id[relation_name]).fillna(0).astype(int)
        one_mask = sampled_df[relation_name] == 1
        for idx in sampled_df.index[one_mask]:
            labels = ensure_list(sampled_df.at[idx, "multi_labels"])
            if relation_name not in labels:
                labels.append(relation_name)
            sampled_df.at[idx, "multi_labels"] = labels

    sampled_df.to_csv(save_path, index=False)
    print(f"[INFO] 재사용분 저장 완료: {save_path}")

    if not new_relations:
        return sampled_df, reused_relations, new_relations

    total_tasks = len(sampled_df) * len(new_relations)
    pbar = tqdm(total=total_tasks, desc="Binary querying ChatGPT")

    with ThreadPoolExecutor(max_workers=BINARY_MAX_WORKERS) as executor:
        for rel_idx, relation_name in enumerate(new_relations, start=1):
            print(f"\n[INFO] Processing NEW relation ({rel_idx}/{len(new_relations)}): {relation_name}")
            if relation_name not in sampled_df.columns:
                sampled_df[relation_name] = pd.NA

            futures = []
            processed = 0
            for idx, row in sampled_df.iterrows():
                futures.append(executor.submit(call_binary_api, idx, row, relation_name))
                if len(futures) >= BINARY_SAVE_EVERY:
                    for f in as_completed(futures):
                        i, pred_bin = f.result()
                        sampled_df.at[i, relation_name] = pred_bin
                        if pred_bin == 1:
                            labels = ensure_list(sampled_df.at[i, "multi_labels"])
                            if relation_name not in labels:
                                labels.append(relation_name)
                            sampled_df.at[i, "multi_labels"] = labels
                        pbar.update(1)
                        processed += 1
                    sampled_df.to_csv(save_path, index=False)
                    print(f"[INFO] relation={relation_name}, {processed}개 저장 완료")
                    futures = []

            if futures:
                for f in as_completed(futures):
                    i, pred_bin = f.result()
                    sampled_df.at[i, relation_name] = pred_bin
                    if pred_bin == 1:
                        labels = ensure_list(sampled_df.at[i, "multi_labels"])
                        if relation_name not in labels:
                            labels.append(relation_name)
                        sampled_df.at[i, "multi_labels"] = labels
                    pbar.update(1)
                sampled_df.to_csv(save_path, index=False)
                print(f"[INFO] relation={relation_name}, 최종 저장 완료")
    pbar.close()
    return sampled_df, reused_relations, new_relations

In [5]:

def build_intersection_matrix(binary_df, save_path=None):
    multi_df = binary_df.copy()
    if "predicted_relation" not in multi_df.columns and "gpt4_phrase" in multi_df.columns:
        multi_df = multi_df.rename(columns={"gpt4_phrase": "predicted_relation"})
    multi_df["multi_labels"] = multi_df["multi_labels"].apply(ensure_list)

    labels_series = multi_df["multi_labels"].explode().dropna()
    labels_count = labels_series.value_counts().reset_index()
    labels_count.columns = ["predicted_relation", "num_samples"]

    if labels_count.empty:
        labels_count["id_list"] = []
        if save_path is not None:
            labels_count.to_csv(save_path, index=False)
        return labels_count

    relation_to_ids = (
        multi_df.explode("multi_labels")
        .dropna(subset=["multi_labels"])
        .groupby("multi_labels")["id"]
        .apply(list)
        .to_dict()
    )
    labels_count["id_list"] = labels_count["predicted_relation"].map(relation_to_ids)

    relations = labels_count["predicted_relation"].astype(str).tolist()
    id_sets = [set(ensure_list(ids)) for ids in labels_count["id_list"]]
    overlap_data = np.zeros((len(relations), len(relations)), dtype=int)
    for i, ids_i in enumerate(id_sets):
        for j, ids_j in enumerate(id_sets):
            overlap_data[i, j] = len(ids_i & ids_j)

    overlap_df = pd.DataFrame(overlap_data, columns=relations)
    labels_count = pd.concat([labels_count.reset_index(drop=True), overlap_df], axis=1)

    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        labels_count.to_csv(save_path, index=False)
        print(f"Saved intersection matrix: {save_path}")
    return labels_count

In [6]:

def normalize_predicate_list(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return [str(p).strip() for p in x if str(p).strip()]
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                v = ast.literal_eval(s)
                if isinstance(v, list):
                    return [str(p).strip() for p in v if str(p).strip()]
            except Exception:
                pass
        if ";" in s:
            return [p.strip().strip("'").strip('"') for p in s.split(";") if p.strip()]
        return [s] if s else []
    return [str(x).strip()] if str(x).strip() else []


def parse_id_list(x):
    return [str(v).strip() for v in ensure_list(x) if str(v).strip()]


def compute_granularity(labels_count, predicate_df):
    labels_count = labels_count.copy()
    tmp = predicate_df[["id", "predicate"]].copy()
    tmp["predicate"] = tmp["predicate"].apply(normalize_predicate_list)
    tmp["predicate_set"] = tmp["predicate"].apply(set)
    id_to_predset = dict(zip(tmp["id"].astype(str), tmp["predicate_set"]))

    def compute_predicate_ratio_for_class(id_list):
        ids = [str(i) for i in parse_id_list(id_list)]
        n = len(ids)
        if n == 0:
            return {}
        pairs = []
        for _id in ids:
            for p in id_to_predset.get(_id, set()):
                pairs.append((_id, p))
        if not pairs:
            return {}
        pair_df = pd.DataFrame(pairs, columns=["id", "predicate"]).drop_duplicates()
        counts = pair_df.groupby("predicate")["id"].nunique()
        return (counts / n).to_dict()

    def mean_ratio(d):
        if not isinstance(d, dict) or len(d) == 0:
            return pd.NA
        vals = [v for v in d.values() if pd.notna(v)]
        return sum(vals) / len(vals) if vals else pd.NA

    labels_count["predicate_ratio"] = labels_count["id_list"].apply(compute_predicate_ratio_for_class)
    labels_count["IDPPA_no0"] = labels_count["predicate_ratio"].apply(mean_ratio)
    return labels_count


def sort_relations_by_granularity(labels_count, anchor=0.5, fixed_relations=None):
    if not 0 <= anchor <= 1:
        raise ValueError("GRANULARITY_ANCHOR는 0과 1 사이여야 합니다.")

    work_df = (
        labels_count[["predicted_relation", "IDPPA_no0"]]
        .dropna(subset=["IDPPA_no0"])
        .drop_duplicates(subset=["predicted_relation"])
        .reset_index(drop=True)
    )
    if work_df.empty:
        return []

    target = work_df["IDPPA_no0"].quantile(anchor)
    work_df["dist_to_anchor"] = (work_df["IDPPA_no0"] - target).abs()
    
    value_ascending = anchor < 0.5
    sorted_df = work_df.sort_values(
        by=["dist_to_anchor", "IDPPA_no0", "predicted_relation"],
        ascending=[True, value_ascending, True],
    )
    sorted_rel_list = sorted_df["predicted_relation"].tolist()

    fixed_relations = unique_keep_order(fixed_relations or [])
    fixed_in_sorted = [r for r in sorted_rel_list if r in set(fixed_relations)]
    ordered = fixed_in_sorted + [r for r in sorted_rel_list if r not in set(fixed_in_sorted)]
    return ordered

In [7]:

def integrate_by_intersection(binary_df, labels_count, sorted_rel_list, fixed_relations=None, threshold=0.5):
    multi_df = binary_df.copy()
    if "predicted_relation" not in multi_df.columns and "gpt4_phrase" in multi_df.columns:
        multi_df = multi_df.rename(columns={"gpt4_phrase": "predicted_relation"})
    multi_df["multi_labels"] = multi_df["multi_labels"].apply(ensure_list)
    multi_df["integration_col"] = pd.NA

    fixed_relations = unique_keep_order(fixed_relations or [])
    fixed_set = set(fixed_relations)

    
    ordered_base_rel_list = fixed_relations + [r for r in sorted_rel_list if r not in fixed_set]
    ordered_base_rel_list = unique_keep_order(ordered_base_rel_list)

    mask_len1 = multi_df["multi_labels"].apply(lambda x: isinstance(x, list) and len(x) == 1)
    multi_df.loc[mask_len1, "integration_col"] = multi_df.loc[mask_len1, "multi_labels"].apply(lambda x: x[0])

    used_ids = set(multi_df.loc[mask_len1, "id"])
    used_base_rel = set()
    relation_cols = relation_columns_from_intersection(labels_count)

    print("ordered_base_rel_list length:", len(ordered_base_rel_list))
    print("fixed_relations length:", len(fixed_relations))

    for step, rel1 in enumerate(ordered_base_rel_list):
        if rel1 in used_base_rel:
            continue
        base_rows = labels_count[labels_count["predicted_relation"] == rel1]
        if base_rows.empty:
            used_base_rel.add(rel1)
            continue

        remain_mask_global = ~multi_df["id"].isin(used_ids)
        if not remain_mask_global.any():
            print("더 이상 사용할 수 있는 샘플이 없어서 종료")
            break

        base_row = base_rows.iloc[0]
        integration_rel_list = []
        for col in relation_cols:
            if col == rel1 or col not in labels_count["predicted_relation"].values:
                continue
            target_row = labels_count[labels_count["predicted_relation"] == col]
            if target_row.empty or col not in labels_count.columns:
                continue
            base_value = pd.to_numeric(base_row[col], errors="coerce")
            num_samples = pd.to_numeric(target_row["num_samples"].iloc[0], errors="coerce")
            if pd.isna(base_value) or pd.isna(num_samples):
                continue
            if base_value >= num_samples * threshold:
                integration_rel_list.append(col)

        if not integration_rel_list:
            used_base_rel.add(rel1)
            continue

        remain_mask = ~multi_df["id"].isin(used_ids)
        has_integration = multi_df["multi_labels"].apply(lambda labels: any(r in labels for r in integration_rel_list))
        step_mask = remain_mask & has_integration
        if not step_mask.any():
            used_base_rel.add(rel1)
            continue

        has_rel1 = multi_df["multi_labels"].apply(lambda labels: rel1 in labels)
        mask_both = step_mask & has_rel1
        mask_only_integration = step_mask & (~has_rel1) & has_integration

        multi_df.loc[mask_both, "integration_col"] = rel1

        def make_new_phrase(labels):
            included = [r for r in integration_rel_list if r in labels]
            return f"new phrase [{rel1}], {included}"

        multi_df.loc[mask_only_integration, "integration_col"] = multi_df.loc[mask_only_integration, "multi_labels"].apply(make_new_phrase)
        used_ids.update(multi_df.loc[step_mask, "id"])
        used_base_rel.add(rel1)

        if step % 20 == 0:
            print(f"step={step}, base_rel={rel1}, used_ids={len(used_ids)}")

    return multi_df

In [ ]:

def extract_all_values(s):
    if pd.isna(s):
        return None
    parts = re.findall(r"\[([^\]]+)\]", str(s))
    values = []
    for p in parts:
        for v in p.split(","):
            v = v.strip().strip("'").strip('"')
            if v:
                values.append(v)
    return ", ".join(values)


def pack_group_samples(df_group: pd.DataFrame) -> str:
    lines = []
    for i, row in enumerate(df_group.itertuples(index=False), start=1):
        lines.append(f"sample {i} : {row.text}, subject : {row.subject}, object : {row.object}")
    return "\n".join(lines)


def build_group_prompt(packed_samples: str, candidate_relation: str, integration_col_list) -> str:
    integration_str = "\n".join([f"- {r}" for r in integration_col_list])
    return (
        "You are an expert annotator for relation extraction tasks.\n"
        "All given samples belong to the same relation class.\n\n"
        "Your task:\n"
        "Decide whether the relation can be expressed using ONE of the EXISTING relations below.\n\n"
        "If YES:\n"
        "existing relation: <EXACT_RELATION_FROM_THE_LIST>\n\n"
        "If NO:\n"
        "Create a NEW relation phrase (5 words or fewer).\n\n"
        "Rules:\n"
        "1) If using an existing relation, you MUST copy it EXACTLY from the list.\n"
        "2) If creating a new relation, output ONLY the phrase (no prefix).\n"
        "3) Do not output anything else.\n\n"
        f"Current candidate relation classes: {candidate_relation}\n\n"
        "EXISTING relation list:\n"
        f"{integration_str}\n\n"
        "Samples:\n"
        f"{packed_samples}\n\n"
        "Output (one line only):\n"
    )


def call_new_phrase_api(df_group: pd.DataFrame, integration_col_list):
    if client is None:
        raise RuntimeError("OPENAI_API_KEY가 설정되지 않았습니다.")
    candidate_relation = df_group["integration_col_from"].iloc[0]
    prompt = build_group_prompt(pack_group_samples(df_group), candidate_relation, integration_col_list)
    result = client.responses.create(model=OPENAI_MODEL, input=prompt, temperature=API_TEMPERATURE)
    return (result.output_text or "").strip()


def generate_new_phrases(multi_df, existing_relation_list, save_path):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    new_df = multi_df.loc[multi_df["integration_col"].astype(str).str.contains("new", na=False)].copy()
    if new_df.empty:
        empty = new_df.copy()
        empty["integration_col_from"] = []
        empty["new_phrase"] = []
        empty.to_csv(save_path, index=False)
        return empty, []

    new_df["integration_col_from"] = new_df["integration_col"].apply(extract_all_values)
    new_df["new_phrase"] = pd.NA

    integration_col_list = unique_keep_order(existing_relation_list)
    integration_set = set(integration_col_list)
    existing_prefix_pat = re.compile(r"^\s*existing\s+relation\s*:\s*", flags=re.IGNORECASE)

    group_phrase_map = {}
    errors = []
    groups = list(new_df.groupby("integration_col", sort=False))

    for group_key, df_group in tqdm(groups, total=len(groups), desc="Group API rethink", miniters=20):
        try:
            out = call_new_phrase_api(df_group, integration_col_list)
            if not out:
                errors.append({"integration_col": group_key, "error": "Empty output"})
                continue

            s = out.strip()
            if existing_prefix_pat.search(s):
                rel = existing_prefix_pat.sub("", s).strip()
                group_phrase_map[group_key] = rel
                continue

            new_rel = s
            group_phrase_map[group_key] = new_rel
            if new_rel and new_rel not in integration_set:
                integration_col_list.append(new_rel)
                integration_set.add(new_rel)
        except Exception as e:
            errors.append({"integration_col": group_key, "error": str(e)})

    new_df["new_phrase"] = new_df["integration_col"].map(group_phrase_map)
    new_df.to_csv(save_path, index=False)
    if errors:
        pd.DataFrame(errors).to_csv(save_path.with_name(save_path.stem + "_errors.csv"), index=False)

    discovered = [r for r in unique_keep_order(new_df["new_phrase"].dropna().tolist()) if r not in set(existing_relation_list)]
    print("discovered new phrases:", len(discovered))
    return new_df, discovered


def compute_frequency_lists(multi_df, new_phrase_df, fixed_before=None, top_ratio=0.1):
    df_not_new = multi_df.loc[~multi_df["integration_col"].astype(str).str.contains("new", na=False)].copy()
    series_parts = []
    if not new_phrase_df.empty and "new_phrase" in new_phrase_df.columns:
        series_parts.append(new_phrase_df["new_phrase"].dropna())
    if not df_not_new.empty:
        series_parts.append(df_not_new["integration_col"].dropna())

    if series_parts:
        combined_series = pd.concat(series_parts, ignore_index=True)
        combined_value_counts = combined_series.value_counts()
    else:
        combined_value_counts = pd.Series(dtype=int)

    
    top_k = int(len(combined_value_counts) * top_ratio)
    fixed_after = combined_value_counts.sort_values(ascending=False).head(top_k).index.tolist()
    fixed_before_set = set(unique_keep_order(fixed_before or []))
    newly_fixed = [r for r in fixed_after if r not in fixed_before_set]
    return combined_value_counts, unique_keep_order(fixed_after), newly_fixed


def b3_precision_recall_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    true_groups = {}
    pred_groups = {}

    for i, (t, p_) in enumerate(zip(y_true, y_pred)):
        true_groups.setdefault(t, set()).add(i)
        pred_groups.setdefault(p_, set()).add(i)

    p_sum = 0.0
    r_sum = 0.0
    n = len(y_true)

    for i, (t, p_) in enumerate(zip(y_true, y_pred)):
        true_set = true_groups[t]
        pred_set = pred_groups[p_]
        inter = len(true_set & pred_set)
        p_sum += inter / len(pred_set) if len(pred_set) > 0 else 0.0
        r_sum += inter / len(true_set) if len(true_set) > 0 else 0.0

    precision = p_sum / n if n else 0.0
    recall = r_sum / n if n else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def evaluate_assignment(df, pred_col="integration_col", label=""):
    from scipy.optimize import linear_sum_assignment
    from sklearn.metrics import adjusted_rand_score, completeness_score, confusion_matrix, homogeneity_score, v_measure_score

    non_na_df = df.loc[~df[pred_col].isna()].copy()
    print(f"[{label}] evaluated rows: {non_na_df.shape}")
    if non_na_df.empty:
        metrics = {"Purity": np.nan, "ARI": np.nan, "V-F1": np.nan, "B3-F1": np.nan, "ACC": np.nan}
        print(f"[{label}] 평가할 할당 결과가 없습니다.")
        return metrics

    result_df = non_na_df
    overall_purity = (
        result_df
        .groupby(pred_col)["relation"]
        .value_counts()
        .groupby(level=0)
        .apply(lambda x: x.max())
        .sum()
        / len(result_df)
    )

    y_true = result_df["relation"].astype(str).values
    y_pred = result_df[pred_col].astype(str).values

    ari = adjusted_rand_score(y_true, y_pred)
    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v_f1 = v_measure_score(y_true, y_pred)
    b3_p, b3_r, b3_f1 = b3_precision_recall_f1(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    hungarian_acc = cm[row_ind, col_ind].sum() / cm.sum()

    print(f"[{label}] Purity : {overall_purity:.4f}")
    print(f"[{label}] ACC    : {hungarian_acc:.4f}")
    print(f"[{label}] B3-F1  : {b3_f1:.4f}")
    print(f"[{label}] V-F1   : {v_f1:.4f}")
    print(f"[{label}] ARI    : {ari:.4f}")

    return {
        "Purity": overall_purity,
        "ARI": ari,
        "V-hom": hom,
        "V-com": com,
        "V-F1": v_f1,
        "B3-P": b3_p,
        "B3-R": b3_r,
        "B3-F1": b3_f1,
        "ACC": hungarian_acc,
    }


def apply_new_phrase_assignments(multi_df, new_phrase_df):
    eval_df = multi_df.copy()
    eval_df["integration_col_after_new_phrase"] = eval_df["integration_col"]
    if new_phrase_df.empty or "new_phrase" not in new_phrase_df.columns:
        return eval_df

    phrase_map = (
        new_phrase_df
        .dropna(subset=["integration_col", "new_phrase"])
        .drop_duplicates(subset=["integration_col"])
        .set_index("integration_col")["new_phrase"]
        .to_dict()
    )
    new_mask = eval_df["integration_col"].astype(str).str.contains("new", na=False)
    eval_df.loc[new_mask, "integration_col_after_new_phrase"] = (
        eval_df.loc[new_mask, "integration_col"].map(phrase_map).fillna(eval_df.loc[new_mask, "integration_col"])
    )
    return eval_df


In [9]:

def run_one_cycle(cycle_idx, previous_state):
    print(f"\n========== Cycle {cycle_idx} ==========")
    cycle_tag = f"{RUN_ID_PREFIX}{cycle_idx}"
    cycle_output_dir = OUTPUT_DIR / cycle_tag
    cycle_output_dir.mkdir(parents=True, exist_ok=True)

    df_result = pd.read_csv(DATA_PATH)
    fixed_relations = previous_state.get("fixed_relations", []) if previous_state else []
    previous_relation_list = previous_state.get("relation_list", []) if previous_state else []

    if cycle_idx == 1:
        binary_df = pd.read_csv(INITIAL_BINARY_PATH)
        labels_count = pd.read_csv(INITIAL_INTERSECTION_PATH)
        print("[Cycle 1] 기존 initial binary/intersection 결과를 사용합니다.")
    else:
        binary_before_df = previous_state["binary_df"]
        candidate_relations = unique_keep_order(previous_relation_list)
        if not candidate_relations:
            raise ValueError("두 번째 사이클부터는 이전 relation_list가 필요합니다.")
        binary_save_path = cycle_output_dir / f"{cycle_tag}_1_binary_question_result_GPT4.csv"
        binary_df, reused_relations, new_query_relations = run_binary_query_for_new_relations(
            df_result=df_result,
            candidate_relations=candidate_relations,
            binary_before_df=binary_before_df,
            save_path=binary_save_path,
        )
        labels_count = build_intersection_matrix(binary_df, cycle_output_dir / f"{cycle_tag}_2_intersection_matrix.csv")
        print(f"[Cycle {cycle_idx}] 새로 질의한 relation 수: {len(new_query_relations)}")

    predicate_df = pd.read_csv(INITIAL_PREDICATE_PATH)
    labels_count = compute_granularity(labels_count, predicate_df)
    labels_count.to_csv(cycle_output_dir / f"{cycle_tag}_3_intersection_matrix_with_granularity.csv", index=False)

    sorted_rel_list = sort_relations_by_granularity(
        labels_count,
        anchor=GRANULARITY_ANCHOR,
        fixed_relations=fixed_relations if cycle_idx > 1 else [],
    )
    with open(cycle_output_dir / f"{cycle_tag}_4_sorted_list_by_IDPPA.json", "w", encoding="utf-8") as f:
        json.dump(sorted_rel_list, f, ensure_ascii=False, indent=2)
    print("sorted_rel_list length:", len(sorted_rel_list))

    multi_df = integrate_by_intersection(
        binary_df=binary_df,
        labels_count=labels_count,
        sorted_rel_list=sorted_rel_list,
        fixed_relations=fixed_relations if cycle_idx > 1 else [],
        threshold=INTEGRATION_THRESHOLD,
    )
    multi_df.to_csv(cycle_output_dir / f"{cycle_tag}_5_integrated.csv", index=False)
    metrics_before_new_phrase = evaluate_assignment(
        multi_df,
        pred_col="integration_col",
        label=f"{cycle_tag} before new phrase generation",
    )

    not_new_list = unique_keep_order(
        multi_df.loc[~multi_df["integration_col"].astype(str).str.contains("new", na=False), "integration_col"].dropna().tolist()
    )
    save_list(not_new_list, cycle_output_dir / f"{cycle_tag}_6_integration_col_list.txt")

    existing_for_rethink = unique_keep_order(not_new_list + (fixed_relations if cycle_idx > 1 else []))
    new_phrase_df, discovered_new_phrases = generate_new_phrases(
        multi_df=multi_df,
        existing_relation_list=existing_for_rethink,
        save_path=cycle_output_dir / f"{cycle_tag}_7_new_phrase_grouped_GPT4.csv",
    )
    save_list(discovered_new_phrases, cycle_output_dir / f"{cycle_tag}_8_new_phrase_list.txt")

    eval_after_new_phrase_df = apply_new_phrase_assignments(multi_df, new_phrase_df)
    eval_after_new_phrase_df.to_csv(cycle_output_dir / f"{cycle_tag}_8_1_assignment_after_new_phrase.csv", index=False)
    metrics_after_new_phrase = evaluate_assignment(
        eval_after_new_phrase_df,
        pred_col="integration_col_after_new_phrase",
        label=f"{cycle_tag} after new phrase generation",
    )

    relation_list = unique_keep_order(existing_for_rethink + new_phrase_df.get("new_phrase", pd.Series(dtype=str)).dropna().tolist())
    save_list(relation_list, cycle_output_dir / f"{cycle_tag}_9_all_relation_list.txt")

    combined_value_counts, fixed_after, newly_fixed = compute_frequency_lists(
        multi_df=multi_df,
        new_phrase_df=new_phrase_df,
        fixed_before=fixed_relations,
        top_ratio=FIXED_TOP_RATIO,
    )
    combined_value_counts.to_csv(cycle_output_dir / f"{cycle_tag}_10_phrase_frequency.csv", header=["count"])
    fixed_ratio_label = str(FIXED_TOP_RATIO).replace(".", "p")
    save_list(fixed_after, cycle_output_dir / f"{cycle_tag}_11_fixed_top_ratio_{fixed_ratio_label}_list.txt")
    save_list(newly_fixed, cycle_output_dir / f"{cycle_tag}_12_newly_fixed_top_ratio_{fixed_ratio_label}_list.txt")

    newly_discovered_set = set(discovered_new_phrases)
    print(f"Cycle {cycle_idx} summary")
    print("  relation_list:", len(relation_list))
    print("  fixed_after:", len(fixed_after))
    print("  newly_fixed_this_cycle:", len(newly_fixed))
    print("  discovered_new_phrases:", len(newly_discovered_set))

    return {
        "cycle_idx": cycle_idx,
        "binary_df": binary_df,
        "labels_count": labels_count,
        "multi_df": multi_df,
        "new_phrase_df": new_phrase_df,
        "relation_list": relation_list,
        "fixed_relations": fixed_after,
        "discovered_new_phrases": unique_keep_order(discovered_new_phrases),
    }

In [10]:

import sys
from datetime import datetime

class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

cycle_states = []
previous_state = None
cycle_idx = 1
previous_relation_list_len = None
unchanged_relation_list_count = 0
log_path.parent.mkdir(parents=True, exist_ok=True)

original_stdout = sys.stdout
original_stderr = sys.stderr

with open(log_path, "a", encoding="utf-8", buffering=1) as log_file:
    sys.stdout = Tee(original_stdout, log_file)
    sys.stderr = Tee(original_stderr, log_file)
    try:
        print(f"\n===== B_2_C_D run started at {datetime.now().isoformat(timespec='seconds')} =====")
        print("log_path:", log_path)

        while True:
            if MAX_CYCLES is not None and cycle_idx > MAX_CYCLES:
                print(f"MAX_CYCLES={MAX_CYCLES}에 도달하여 종료합니다.")
                break

            state = run_one_cycle(cycle_idx, previous_state)
            cycle_states.append(state)

            current_relation_list_len = len(state.get("relation_list", []))
            if previous_relation_list_len is not None and current_relation_list_len == previous_relation_list_len:
                unchanged_relation_list_count += 1
            else:
                unchanged_relation_list_count = 0

            print("relation_list length unchanged consecutive count:", unchanged_relation_list_count)

            if unchanged_relation_list_count >= 2:
                print("relation_list 수가 2사이클 연속으로 변화 없어 종료합니다.")
                break

            discovered = state.get("discovered_new_phrases", [])
            if STOP_WHEN_NO_NEW_PHRASES and len(discovered) == 0:
                print(f"Cycle {cycle_idx}에서 rethink 결과 새로운 phrase가 발견되지 않아 종료합니다.")
                break

            previous_relation_list_len = current_relation_list_len
            previous_state = state
            cycle_idx += 1

        print("완료된 cycle 수:", len(cycle_states))
        print(f"===== B_2_C_D run finished at {datetime.now().isoformat(timespec='seconds')} =====")
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr



===== B_2_C_D run started at 2026-05-22T01:16:59 =====
log_path: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/TACRED_full_cycle_log.txt

========== Cycle 1 ==========
[Cycle 1] 기존 initial binary/intersection 결과를 사용합니다.
sorted_rel_list length: 930
ordered_base_rel_list length: 930
fixed_relations length: 0
step=0, base_rel=member_of_religious_organization, used_ids=88
step=20, base_rel=family_relation_father_son, used_ids=1001
step=80, base_rel=accused_of, used_ids=1571
step=320, base_rel=held_position, used_ids=1804
step=620, base_rel=statement_about_behavior, used_ids=1882
[Cycle1 before new phrase generation] evaluated rows: (1892, 955)
[Cycle1 before new phrase generation] Purity : 0.6348
[Cycle1 before new phrase generation] ACC    : 0.3658
[Cycle1 before new phrase generation] B3-F1  : 0.3961
[Cycle1 before new phrase generation] V-F1   : 0.6473
[Cycle1 before new phrase generation] ARI    : 0.3324
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle1/Cycl

Group API rethink: 100%|##########| 344/344 [06:54<00:00,  1.21s/it]


discovered new phrases: 78
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle1/Cycle1_8_new_phrase_list.txt (78)
[Cycle1 after new phrase generation] evaluated rows: (1892, 956)
[Cycle1 after new phrase generation] Purity : 0.5412
[Cycle1 after new phrase generation] ACC    : 0.4059
[Cycle1 after new phrase generation] B3-F1  : 0.3976
[Cycle1 after new phrase generation] V-F1   : 0.6160
[Cycle1 after new phrase generation] ARI    : 0.3395
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle1/Cycle1_9_all_relation_list.txt (157)
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle1/Cycle1_11_fixed_top_ratio_0p1_list.txt (15)
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle1/Cycle1_12_newly_fixed_top_ratio_0p1_list.txt (15)
Cycle 1 summary
  relation_list: 157
  fixed_after: 15
  newly_fixed_this_cycle: 15
  discovered_new_phrases: 78
relation_list length unchanged consecutive count: 0

========== Cycle 2 =========

/tmp/ipykernel_1033173/425311638.py:110: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sampled_df[relation_name] = sampled_df["id"].map(before_by_id[relation_name]).fillna(0).astype(int)
/tmp/ipykernel_1033173/425311638.py:110: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sampled_df[relation_name] = sampled_df["id"].map(before_by_id[relation_name]).fillna(0).astype(int)
/tmp/ipykernel_1033173/425311638.py:110: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

[INFO] 재사용분 저장 완료: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle2/Cycle2_1_binary_question_result_GPT4.csv
Saved intersection matrix: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle2/Cycle2_2_intersection_matrix.csv
[Cycle 2] 새로 질의한 relation 수: 0
sorted_rel_list length: 157
ordered_base_rel_list length: 157
fixed_relations length: 15
step=60, base_rel=Ownership-Interest-Bid, used_ids=1257
[Cycle2 before new phrase generation] evaluated rows: (1844, 179)
[Cycle2 before new phrase generation] Purity : 0.5819
[Cycle2 before new phrase generation] ACC    : 0.3986
[Cycle2 before new phrase generation] B3-F1  : 0.3814
[Cycle2 before new phrase generation] V-F1   : 0.6356
[Cycle2 before new phrase generation] ARI    : 0.3374
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle2/Cycle2_6_integration_col_list.txt (60)


Group API rethink: 100%|##########| 130/130 [02:29<00:00,  1.15s/it]


... (27 outputs left out)

Group API rethink: 100%|##########| 84/84 [01:36<00:00,  1.15s/it]


discovered new phrases: 27
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle8/Cycle8_8_new_phrase_list.txt (27)
[Cycle8 after new phrase generation] evaluated rows: (1643, 87)
[Cycle8 after new phrase generation] Purity : 0.3475
[Cycle8 after new phrase generation] ACC    : 0.3025
[Cycle8 after new phrase generation] B3-F1  : 0.3679
[Cycle8 after new phrase generation] V-F1   : 0.5650
[Cycle8 after new phrase generation] ARI    : 0.1906
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle8/Cycle8_9_all_relation_list.txt (64)
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle8/Cycle8_11_fixed_top_ratio_0p1_list.txt (6)
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle8/Cycle8_12_newly_fixed_top_ratio_0p1_list.txt (1)
Cycle 8 summary
  relation_list: 64
  fixed_after: 6
  newly_fixed_this_cycle: 1
  discovered_new_phrases: 27
relation_list length unchanged consecutive count: 1

========== Cycle 9 ==========
[INFO

Group API rethink: 100%|##########| 84/84 [01:33<00:00,  1.12s/it]


discovered new phrases: 27
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle9/Cycle9_8_new_phrase_list.txt (27)
[Cycle9 after new phrase generation] evaluated rows: (1643, 87)
[Cycle9 after new phrase generation] Purity : 0.3475
[Cycle9 after new phrase generation] ACC    : 0.3031
[Cycle9 after new phrase generation] B3-F1  : 0.3700
[Cycle9 after new phrase generation] V-F1   : 0.5745
[Cycle9 after new phrase generation] ARI    : 0.2099
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle9/Cycle9_9_all_relation_list.txt (64)
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle9/Cycle9_11_fixed_top_ratio_0p1_list.txt (6)
Saved list: /workspace/KG-known0/ARR2026/MINT-ORE/output_g05d07/Cycle9/Cycle9_12_newly_fixed_top_ratio_0p1_list.txt (0)
Cycle 9 summary
  relation_list: 64
  fixed_after: 6
  newly_fixed_this_cycle: 0
  discovered_new_phrases: 27
relation_list length unchanged consecutive count: 2
relation_list 수가 2사이클 연속으로 변화 없어 종료합

In [11]:

final_state = cycle_states[-1] if cycle_states else None
if final_state is not None:
    print("final cycle:", final_state["cycle_idx"])
    print("final relation_list length:", len(final_state["relation_list"]))
    print("final fixed_relations length:", len(final_state["fixed_relations"]))
    print("final discovered_new_phrases length:", len(final_state["discovered_new_phrases"]))
    display(final_state["multi_df"].head(2))

final cycle: 9
final relation_list length: 64
final fixed_relations length: 6
final discovered_new_phrases length: 27


,id,relation,text,subject,subject_type,object,object_type,subj_start,subj_end,obj_start,...,location_of_medical_diagnosis,died_in_location,place_of_residence,death_year,signed_record_deal_with,romantic_association,graduated_from,spouse_of,birthplace or hometown,integration_col
0,61b3aa9e36d5d8427e8e,org:alternate_names,"DC : Nov 30th thru Dec 2nd , Servicemembers Un...",Servicemembers United,ORGANIZATION,Servicemembers Legal Defense Network,ORGANIZATION,8,9,28,...,0,0,0,0,0,0,0,0,0,collaborators
1,61b3aa9e368a0e3dfbd2,org:alternate_names,With the tally completed in more than half the...,All Basotho Convention,ORGANIZATION,ABC,ORGANIZATION,29,31,33,...,0,0,0,0,0,0,0,0,0,League_Type
